# ДЗ-2. Витрина клиента

Вы — аналитик ПРАЙМ, и у вас свой **сегмент клиентов**: его прислал бот 26 сентября, он же лежит в вашей строке `prime.assignments`, в колонке `client_filter`. Окно фактов у всех общее — первое полугодие 2026 года.

Задача недели — собрать **витрину клиента**: таблицу, в которой одна строка — один клиент, а рядом всё, что мы про него знаем. Из витрины потом считается всё остальное, поэтому лишние деньги в ней становятся лишними везде.

Идём по лестнице. Сначала вы дописываете один аргумент, как на семинаре, потом целую строку, потом пишете задачу сами, а в конце собираете витрину и доказываете протоколом, что двойного счёта нет.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение, ваш сегмент, выгрузка восьми таблиц | запускаете |
| **2. Уровень 1** | задачи 1–4: не хватает одного аргумента | вписываете аргумент вместо `ЗАПОЛНИТЕ` |
| **3. Уровень 2** | задачи 5–8: не хватает строки | дописываете строку |
| **4. Уровень 3** | задачи 9–12 | пишете код целиком |
| **5. Уровень 4** | задача 13: витрина и протокол сборки | пишете код целиком |
| **6. Итог** | готовый код: ваши ответы и проверка их формата | запускаете перед сдачей |
| **7. Исследование** | две части для тех, кто хочет 9 или 10 | по желанию |

После каждого уровня — вопрос, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N».

**`ЗАПОЛНИТЕ`** — место, куда вписать ваш код. Пока оно не заполнено, ячейка падает с `NameError: name 'ЗАПОЛНИТЕ' is not defined` — так и задумано.

**Оценка.** Части 1–6 дают до 8 баллов: сделали всё аккуратно — 8, и это «отлично». 9 и 10 ставятся только за исследование в части 7. Все критерии и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-2-витрина-клиента).

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей выполните **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в вашем репозитории как `notebooks/hw2.ipynb`, JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно: запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Подключение | подключается к базе через ваш `.env` и заводит функцию `q(sql)` |
| 1.2 Ваш сегмент | достаёт условие `client_filter` из вашей строки `prime.assignments` |
| 1.3 Выгрузка | загружает восемь таблиц — дальше всё в pandas |
| 1.4 Формат ответов | заводит служебную функцию для итога |

Весь SQL задания собран здесь, в готовом коде. Дальше вы работаете только в pandas.

### 1.1 Подключение

Функция `q(sql)` отправляет запрос в базу и возвращает `DataFrame`, как на семинарах. Соединение открывается на каждый запрос и сразу закрывается.

Если здесь ошибка `Пакет prime не найден`, ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN`, в корне `prime-monitor` нет файла `.env`.

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

### 1.2 Ваш сегмент

Сегмент — это клиенты, которые встречаются в вашей таблице с вашим признаком за ваш период. Бот прислал его вам 26 сентября готовым условием для `where`, оно же лежит в вашей строке `prime.assignments`.

**Эту ячейку не меняйте.** При проверке преподаватель подставит сюда условие **другого** сегмента, и все ответы должны пересчитаться сами. Поэтому ниже нигде не вписывайте руками ни условие, ни числа своего сегмента.

In [ ]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(CLIENT_FILTER)

### 1.3 Выгрузка

Восемь таблиц: первые пять — про ваш сегмент, последние три — справочник тарифов и данные всей базы.

| Переменная | Одна строка — это | Что внутри |
|---|---|---|
| `clients` | клиент сегмента | `client_id`, дата регистрации, регион, канал привлечения |
| `subs` | подписка, которой владеет клиент сегмента | тариф, даты начала и конца, пробная ли, причина отмены |
| `pay` | попытка списания по этим подпискам за окно фактов | сумма, **статус**, способ оплаты |
| `sub_members` | участник подписки из `subs` | `subscription_id` и `member_id` — кто подключён к подписке |
| `members` | участие клиента сегмента в любой подписке | `subscription_id` и `client_id` — клиент сегмента как участник |
| `tariffs` | цена тарифа в интервале дат | весь справочник, пять строк |
| `spend` | расходы канала за день | `channel`, `spend_date`, `amount` |
| `registrations` | канал × день регистрации | `n_clients` — сколько клиентов всей базы пришло из канала в этот день |

**Окно фактов общее для всех:** с 1 января по 30 июня 2026 года включительно. В запросе правый конец не включён: `paid_at < '2026-07-01'`.

Самые большие сегменты выгружаются около минуты и занимают в памяти до 2 ГБ — закройте лишние вкладки браузера. Если выгрузка идёт дольше двух минут, сервер оборвёт запрос — что делать, написано в тексте задания, раздел «Если что-то не работает».

In [ ]:
WINDOW = ("2026-01-01", "2026-07-01")   # окно фактов: первое полугодие 2026, правый конец не включён
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

sub_members = q(f"""
    select m.subscription_id, m.client_id as member_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

N = len(clients)
print("ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов:", N)
print()
for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("sub_members", sub_members),
                    ("members", members), ("tariffs", tariffs), ("spend", spend),
                    ("registrations", registrations)]:
    print(f"{name:<14} {len(frame):>9} строк")

### 1.4 Формат ответов

Служебная функция: по ней итог в части 6 проверяет, что ответ записан в нужном формате — целое, число с копейками, список. **Верно ли посчитано, она не знает и не сообщает**: это проверяет преподаватель. Код ячейки свёрнут, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 2 знаков — рубли",
    2: "int",
    3: "int",
    4: "float до 2 знаков — рубли (0.0, если таких клиентов нет)",
    5: "float до 2 знаков — рубли",
    6: "float до 2 знаков — рубли",
    7: "int",
    8: "float до 1 знака — проценты",
    9: "int",
    10: "list [float до 2 знаков, int]",
    11: "list [float до 2 знаков, int]",
    12: "list из 3 int",
    13: "list из 3 int — то, что вернула print_protocol()",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:   # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def pct(x) -> bool:     # float, у которого не больше одного знака после запятой
        return type(x) is float and round(x, 1) == x

    def whole(x) -> bool:
        return type(x) is int

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: money, 2: whole, 3: whole, 4: money, 5: money, 6: money, 7: whole, 8: pct,
        9: whole,
        10: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        11: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        12: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
        13: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Уровень 1 — дыра в один аргумент

Как на семинаре: код готов, не хватает одного аргумента. Впишите его вместо `ЗАПОЛНИТЕ` и запустите ячейку.

Последняя строка каждой задачи кладёт ответ в `answer_N`. Эти строки и имена `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом сегменте.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All. Замените всё содержимое ячейки одной строкой `answer_N = None`, где N — номер задачи: итог покажет «❌ не решена», а остальное проверится как обычно.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 6 проверяет только формат. Правильность проверяет преподаватель — на вашем сегменте и на контрольном.

**В `pay` лежат все попытки списания:** `success`, `failed` и `refunded`. Какие из них деньги, решаете вы — это задача 1.

### Задача 1. Контрольное число ДЕНЬГИ

Первое правило сборки витрины: контрольное число считают до сборки. Мы заранее знаем, сколько денег принёс сегмент за окно, — и потом витрина обязана дать ровно столько же.

**Что сделать**

Оставьте в `pay` только платежи, которые действительно принесли деньги, и посчитайте их сумму.

**Формат ответа:** `float` до двух знаков, рубли — например `98765432.1`.

**Подсказка:** Какие статусы бывают, покажет `pay["status"].value_counts()`.

In [ ]:
pay["status"].value_counts()

In [ ]:
# Какие попытки списания — это деньги, которые остались у компании?
success = pay[pay["status"] == "success"]
revenue = success["amount"].sum()

print("платежей с этим статусом:", len(success))
answer_1 = round(float(revenue), 2)
answer_1

### Задача 2. Подписки без платежа

Руководитель спрашивает, сколько подписок сегмента за полгода не принесли ни рубля. Чтобы ответить, к каждой подписке присоединяют число её успешных платежей и смотрят, у кого пусто.

**Что сделать**

Выберите вид соединения так, чтобы подписки без платежей **остались** в результате — с пропуском в `n_payments`.

**Формат ответа:** `int` — например `4321`.

**Подсказка:** Слайд 7 Л4: `how=` решает судьбу строк, которым не нашлась пара.

In [ ]:
    # Сколько успешных платежей пришло по каждой подписке за окно.
    by_sub = (pay[pay["status"] == "success"]
              .groupby("subscription_id", as_index=False)
              .agg(n_payments=("payment_id", "size")))
    
    # Подписка без платежей должна остаться в результате — с пропуском в n_payments.
    subs_pay = subs.merge(by_sub, on="subscription_id", how="left", validate="1:1")
    
    no_payment = subs_pay["n_payments"].isna().sum()
    print("подписок:", len(subs), " строк после соединения:", len(subs_pay))
    answer_2 = int(no_payment)
    answer_2

### Задача 3. Кто платил

Сколько владельцев из сегмента заплатили за полгода хотя бы раз? Для этого к каждому платежу приписывают владельца подписки. Но прежде чем соединять, объявляют кратность: сколько платежей приходится на одну подписку и сколько подписок — на один платёж.

**Что сделать**

Впишите в `validate=` кратность связи «платёж → подписка».

**Формат ответа:** `int` — например `5432`.

**Подсказка:** Слайд 6 Л4, четыре случая кратности. Если слева на один ключ приходится много строк, это «m».

In [ ]:
success = pay[pay["status"] == "success"]

# Сколько платежей приходится на одну подписку — и сколько подписок на один платёж?
pay_owner = success.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id", how="left", validate="m:1",
)

payers = pay_owner["owner_client_id"].nunique()
print("платежей до соединения:", len(success), " после:", len(pay_owner))
answer_3 = int(payers)
answer_3

### Задача 4. Выручка без канала

Маркетинг смотрит выручку по каналам привлечения. У части клиентов канал не записан, и если этого не заметить, сумма по каналам тихо разойдётся с общей выручкой.

**Что сделать**

Сделайте так, чтобы клиенты без канала попали в отчёт отдельной строкой, и посчитайте их выручку.

**Формат ответа:** `float` до двух знаков, рубли — например `123456.7`. Если таких клиентов в сегменте нет — `0.0`.

**Подсказка:** Слайд 18 Л4: сумма по группам не сходится с общей. Сверьте сумму по каналам с выручкой задачи 1.

In [ ]:
success = pay[pay["status"] == "success"]
pay_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="m:1"))

# Клиенты без канала должны остаться в отчёте отдельной строкой.
by_channel = pay_client.groupby("acquisition_channel", dropna=False)["amount"].sum()

print(by_channel.round(2).to_string())
no_channel = by_channel[by_channel.index.isna()].sum()
answer_4 = round(float(no_channel), 2)
answer_4

**Вопрос 1.** В `pay` есть попытки списания со статусом `refunded`, и сумма у них положительная. Сколько таких строк в вашем сегменте и на какую сумму? Почему эту сумму нельзя прибавить к выручке — и почему её нельзя вычесть из выручки?

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
refunded = pay[pay["status"] == "refunded"]
refunded_count = len(refunded)
refunded_amount = refunded["amount"].sum()
print("Строк со статусом refunded:", refunded_count)
print("Сумма по этим строкам, ₽:", round(float(refunded_amount), 2))
print("Выручка только по success, ₽:", answer_1)

#### ✍️ Ответ на вопрос 1

Количество попыток списания со статусом refunded - 2 902 на сумму 892 998 руб. Эту сумму нельзя прибавить, потому что эта сумма была уплачена клиентом и возвращена, то есть итог - 0 руб. Вычесть нельзя, потому что в выручке уже исключены эти платежи фильтром.

## Часть 3. Уровень 2 — дыра в строку

Теперь не хватает целой строки: выражения, условия или набора аргументов. Всё остальное готово. Правила те же: строки с `answer_N` не трогайте, не получилось — `answer_N = None`.

### Задача 5. Свернуть до клиента

Средний доход с платящего клиента — ARPPU — это выручка, делённая на число клиентов, которые платили. Чтобы посчитать его по витрине, платежи сначала сворачивают до зерна «клиент» и только потом соединяют с клиентами — иначе соединение перемножит строки.

**Что сделать**

Допишите `agg(...)`: для каждого владельца — `n_payments` (сколько успешных платежей) и `paid` (сколько он заплатил).

**Формат ответа:** `float` до двух знаков, рубли — например `1234.56`.

**Подсказка:** Слайд 17 Л4, именованные агрегаты: `agg(имя=("колонка", "функция"), ...)`.

In [ ]:
success = pay[pay["status"] == "success"]

# Одна строка — один владелец: сколько успешных платежей и сколько денег.
by_client = (success
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=("payment_id","size"),paid=("amount","sum")))

mart5 = clients.merge(by_client, left_on="client_id", right_on="owner_client_id",
                      how="left", validate="1:1")

paying = mart5["n_payments"].notna().sum()
arppu = mart5["paid"].sum() / paying
print("клиентов:", len(mart5), " из них платили:", paying)
answer_5 = round(float(arppu), 2)
answer_5

### Задача 6. Цена по справочнику

В `tariffs` лежит история цен: у тарифа несколько строк с интервалами действия, а одна строка ошибочная — вы нашли её на семинаре. Сколько принёс тариф «Расширенный» по цене справочника?

**Что сделать**

Допишите две строки: 1) справочник без ошибочной строки; 2) условие «момент платежа попадает в интервал действия цены» по конвенции курса.

**Формат ответа:** `float` до двух знаков, рубли — например `2345678.9`.

**Подсказка:** Раздел 2 семинара С4. Конвенция границы — `(paid_at >= valid_from) & (paid_at < valid_to + 1 день)`, потому что `valid_to` — это дата, то есть полночь. Каждый платёж должен найти в справочнике ровно одну цену: сравните число строк до и после.

In [ ]:
# Даты справочника — дни без часового пояса, а платежи — моменты в UTC.
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Справочник без строки, из-за которой интервалы одного тарифа перекрываются.
ref_fixed = ref[ref["monthly_fee"] != 429]

pay_t = (pay[pay["status"] == "success"]
         .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
m = pay_t.merge(ref_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]],
                on="tariff_id")

# 2. Момент платежа попадает в интервал действия цены — по конвенции курса.
hit = (m["paid_at"] >= m["valid_from"]) & (m["paid_at"] < m["valid_to"] + pd.Timedelta(days=1))
priced = m[hit]
print("платежей:", len(pay_t), " строк с ценой:", len(priced))
extended = priced.loc[priced["tariff_name"] == "Расширенный", "monthly_fee"].sum()
answer_6 = round(float(extended), 2)
answer_6

In [ ]:
print(tariffs.to_string(index=False))

### Задача 7. Выше среднего по своему каналу

Маркетинг хочет знать, сколько платящих клиентов приносят больше, чем типичный клиент **их же** канала. Для этого рядом с каждым клиентом ставят среднее по его каналу — не сворачивая таблицу.

**Что сделать**

Допишите строку: `channel_avg` — среднее `paid` по каналу клиента, по значению в каждой строке.

**Формат ответа:** `int` — например `6543`.

**Подсказка:** Слайд 19 Л4: `transform` возвращает столько же строк, сколько было.

In [ ]:
success = pay[pay["status"] == "success"]
per_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .groupby("owner_client_id", as_index=False)
              .agg(paid=("amount", "sum"))
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="1:1"))
per_client = per_client[per_client["acquisition_channel"].notna()]   # только известные каналы

# Рядом с каждым клиентом — среднее paid по его каналу. Строк остаётся столько же.
per_client["channel_avg"] = (per_client.groupby("acquisition_channel", dropna=False)["paid"].transform("mean"))
above = (per_client["paid"] > per_client["channel_avg"]).sum()
print("платящих клиентов с известным каналом:", len(per_client))
answer_7 = int(above)
answer_7

### Задача 8. Выручка по месяцам

Финансам нужна таблица: строки — месяцы, колонки — тарифы, в клетках — выручка. По ней видно, как выручка «Базового» менялась от месяца к месяцу.

**Что сделать**

Допишите аргументы `pivot_table`: строки — `month`, колонки — `tariff_id`, значения — сумма `amount`.

**Формат ответа:** `float` до одного знака, проценты — например `87.6`.

**Подсказка:** Слайд 23 Л4.

In [ ]:
success = (pay[pay["status"] == "success"]
           .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

# Строки — месяцы, колонки — тарифы (1 — «Базовый», 2 — «Расширенный»), в клетках — выручка.
by_month = success.pivot_table(index="month", columns="tariff_id", values="amount", aggfunc="sum")

print(by_month.round(2).to_string())
april_to_march = by_month.loc["2026-04", 1] / by_month.loc["2026-03", 1] * 100
answer_8 = round(float(april_to_march), 1)
answer_8

**Вопрос 2.** Выручка «Базового» в апреле — `answer_8` процентов от мартовской. Это отток клиентов? Проверьте по дням: сколько дней апреля есть в `pay` и как выглядит выручка по дням. Объясните, что случилось, и что вы написали бы финансам в отчёте.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
# Все попытки списания в апреле, независимо от статуса.
april = pay[
    (pay["paid_at"] >= pd.Timestamp("2026-04-01", tz="UTC"))
    & (pay["paid_at"] < pd.Timestamp("2026-05-01", tz="UTC"))
].copy()

april["day"] = april["paid_at"].dt.normalize()

# Проверяем, за какие даты есть данные.
all_days = pd.date_range(
    "2026-04-01", "2026-04-30", freq="D", tz="UTC"
)
present_days = pd.DatetimeIndex(april["day"].unique())
missing_days = all_days.difference(present_days)

print("Дней апреля с попытками списания:", april["day"].nunique())
print("Дней в апреле:", len(all_days))
print("Отсутствующие даты:", missing_days.strftime("%Y-%m-%d").tolist())

# Успешные платежи апреля с указанием тарифа.
april_success = (
    april[april["status"] == "success"]
    .merge(
        subs[["subscription_id", "tariff_id"]],
        on="subscription_id",
        validate="m:1"
    )
)

# Выручка «Базового» по дням.
daily = (
    april_success[april_success["tariff_id"] == 1]
    .groupby("day")
    .agg(
        n_payments=("payment_id", "size"),
        revenue=("amount", "sum")
    )
    .reindex(all_days)
)

daily.index.name = "day"

print("\nУспешные платежи и выручка «Базового» по дням:")
print(daily.round(2).to_string())

print("\nАпрельская выручка в процентах от мартовской:", answer_8)

#### ✍️ Ответ на вопрос 2

В данных пропущены платежи за 14 и 15 апреля, что непосредственно влияет на процент апрельской выручки от мартовской (87,1%). В отчете я бы указал, что по этому сравнению нельзя сделать вывод об оттоке клиентов, так как из-за неполноты данных сравнение не отражает ситуацию за апрель.

## Часть 4. Уровень 3 — пишете сами

Заготовок больше нет: в ячейке только условие в комментарии и строка с `answer_N`. Код вы пишете целиком — от загруженных таблиц до ответа. Печатайте промежуточные результаты: так проще найти ошибку и объяснить решение на защите.

Каждая задача опирается на приём с лекции Л4 и семинара С4 — номер слайда стоит в подсказке. Ответ кладите в `answer_N` обычным `int` или `float`, а не `numpy`-числом: `int(...)`, `round(float(...), 2)`.

### Задача 9. Люди, а не места

Сколько людей сегмента пользуются ПРАЙМ? Пользуется тот, кто владеет подпиской или участвует хотя бы в одной чужой. Один человек может участвовать в нескольких подписках: если не свернуть участия до клиента, вы насчитаете места, а не людей.

**Что сделать**

Посчитайте клиентов сегмента, которые владеют подпиской (`subs`) или участвуют хотя бы в одной (`members`). Одна строка — один клиент.

**Формат ответа:** `int` — например `7654`.

**Подсказка:** Раздел 3 семинара С4: свернуть участия до клиента, соединить с `validate="1:1"`, флаг «владелец или участник».

In [ ]:
# Задача 9. Сколько клиентов сегмента пользуются ПРАЙМ: владеют подпиской
# или участвуют хотя бы в одной. Одна строка — один клиент. Ответ — обычный int.
# Сворачиваем участия до одной строки на клиента.
memberships9 = (members.groupby("client_id", as_index=False).agg(n_memberships=("subscription_id", "size")))
print("Клиентов с участиями:", len(memberships9))
display(memberships9.head())

In [ ]:
# Сохраняем всех клиентов, включая тех, у кого нет участий.
mart9 = clients[["client_id"]].merge(memberships9,on="client_id",how="left",validate="1:1")
print("Клиентов до соединения:", len(clients))
print("Строк после соединения:", len(mart9))
print("Клиентов без участий:", int(mart9["n_memberships"].isna().sum()))
display(mart9.head())

In [ ]:
# Отсутствие участия обозначаем целым числом 0.
mart9["n_memberships"] = (mart9["n_memberships"].fillna(0).astype("int64"))
print("Пропусков осталось:", int(mart9["n_memberships"].isna().sum()))
print("Тип количества участий:", mart9["n_memberships"].dtype)
display(mart9.head())

In [ ]:
# Проверяем, входит ли клиент в список владельцев подписок.
mart9["is_owner"] = mart9["client_id"].isin(subs["owner_client_id"])
# Клиент пользуется ПРАЙМ, если он владелец или участник.
mart9["uses_prime"] = (mart9["is_owner"] | (mart9["n_memberships"] > 0))
print("Владельцев:", int(mart9["is_owner"].sum()))
print("Участников:", int((mart9["n_memberships"] > 0).sum()))
print("Одновременно владельцев и участников:",int((mart9["is_owner"] & (mart9["n_memberships"] > 0)).sum()))
display(mart9.head())

In [ ]:
# Считаем каждого пользователя один раз: одна строка — один клиент.
answer_9 = int(mart9["uses_prime"].sum())
print("Всего клиентов:", len(mart9))
print("Пользуются ПРАЙМ:", answer_9)
print("Не пользуются ПРАЙМ:", len(mart9) - answer_9)
answer_9

### Задача 10. Сколько человек на подписку

Подпиской пользуются владелец и те, кого он подключил. Подключить можно не больше `share_slots` человек — это поле справочника тарифов. Сколько в среднем человек приходится на подписку сегмента и сколько подписок заполнены до предела?

**Что сделать**

1. Для каждой подписки из `subs`: один владелец плюс число её строк в `sub_members`.
2. Среднее по всем подпискам сегмента — первая часть ответа.
3. Присоедините к подпискам `share_slots` из `tariffs`. Подписок, где людей ровно `share_slots` + 1, — вторая часть ответа.
4. Напечатайте, у скольких подписок людей больше, чем `share_slots` + 1. Это инвариант: число обязано быть нулём.

**Формат ответа:** `[среднее, заполненных до предела]` — например `[3.21, 987]`: `float` до двух знаков и `int`.

**Подсказка:** Слайды 8 и 14 Л4. У одного `tariff_id` в справочнике несколько строк, и `validate="m:1"` об этом скажет. Какая часть справочника на самом деле нужна для `share_slots`?

In [ ]:
# Задача 10. Люди на подписках сегмента, считая владельца, и лимит share_slots.
# Ответ — [среднее на подписку (float до 2 знаков), заполненных до предела (int)].
from IPython.display import display
# Считаем строки участий по каждой подписке.
members10 = (sub_members.groupby("subscription_id", as_index=False).agg(n_members=("member_id", "size")))
print("Подписок с подключёнными участниками:", len(members10))
display(members10.head())

In [ ]:
# Сохраняем подписки, даже если к ним никто не подключён.
subs10 = subs[["subscription_id", "tariff_id"]].merge(members10,on="subscription_id",how="left",validate="1:1")
print("Подписок до соединения:", len(subs))
print("Строк после соединения:", len(subs10))
print("Без подключённых участников:", int(subs10["n_members"].isna().sum()))
display(subs10.head())

In [ ]:
# Нет подключённых участников — ставим 0; владелец есть у каждой подписки.
subs10["n_members"] = subs10["n_members"].fillna(0).astype("int64")
subs10["n_people"] = subs10["n_members"] + 1
average_people10 = subs10["n_people"].mean()
print("Среднее число людей на подписку:", round(float(average_people10), 2))
display(subs10.head())

In [ ]:
# Берём только тариф и лимит: история цен здесь не нужна.
limits10 = tariffs[["tariff_id", "share_slots"]].drop_duplicates()
print("Лимиты по тарифам:")
display(limits10)
# Одному тарифу справа должна соответствовать ровно одна строка.
subs10 = subs10.merge(limits10,on="tariff_id",how="left",validate="m:1")
print("Строк после добавления лимитов:", len(subs10))
print("Подписок без найденного лимита:", int(subs10["share_slots"].isna().sum()))
display(subs10.head())

In [ ]:
# Лимит share_slots относится к подключённым участникам; добавляем владельца.
subs10["people_limit"] = subs10["share_slots"] + 1
full10 = subs10["n_people"] == subs10["people_limit"]
over_limit10 = subs10["n_people"] > subs10["people_limit"]
print("Подписок, заполненных до предела:", int(full10.sum()))
print("Подписок с превышением лимита:", int(over_limit10.sum()))
# Формат ответа: среднее число людей и количество заполненных подписок.
answer_10 = [round(float(average_people10), 2),int(full10.sum())]
answer_10

### Задача 11. Сколько стоил клиент

CAC — сколько стоило привести одного клиента. Расходы на привлечение известны по каналу и дню, а не по клиенту: присоединить их к витрине напрямую нельзя. Поэтому CAC считают на его собственном зерне — «канал × месяц» — и только потом приписывают клиентам.

**Что сделать**

1. CAC канала за месяц — это расходы канала за месяц (`spend`), делённые на число всех клиентов базы, которые пришли из этого канала в этом месяце (`registrations`). Берите только полные месяцы расходов — с 2025-02 по 2026-07.
2. Каждому клиенту сегмента припишите CAC его канала в месяц его регистрации.
3. Посчитайте средний CAC по клиентам сегмента, у которых он определён, и число клиентов, у которых он не определён: канал не записан или регистрация вне этих месяцев.

**Формат ответа:** `[средний CAC, клиентов без CAC]` — например `[432.1, 876]`: `float` до двух знаков и `int`.

**Подсказка:** Слайд 28 Л4: `marketing_spend` не присоединяется к витрине — у него другое зерно. Месяц из даты: `pd.to_datetime(...).dt.strftime("%Y-%m")`. Соединение по двум колонкам: `on=["channel", "month"]`, и кратность объявите.

In [ ]:
# Задача 11. CAC по каналу и месяцу, затем средний CAC клиентов сегмента.
# Ответ — [средний CAC (float до 2 знаков), клиентов без CAC (int)].
from IPython.display import display

# Работаем с копиями, чтобы не менять исходные таблицы.
spend11 = spend.copy()
registrations11 = registrations.copy()
spend11["month"] = (pd.to_datetime(spend11["spend_date"]).dt.strftime("%Y-%m"))
registrations11["month"] = (pd.to_datetime(registrations11["reg_date"]).dt.strftime("%Y-%m"))
# По условию берём полные месяцы: февраль 2025 — июль 2026.
spend11 = spend11[(spend11["month"] >= "2025-02") & (spend11["month"] <= "2026-07")]
registrations11 = registrations11[(registrations11["month"] >= "2025-02") & (registrations11["month"] <= "2026-07")]
print("Строк расходов в периоде:", len(spend11))
print("Строк регистраций в периоде:", len(registrations11))
display(spend11.head())

In [ ]:
# Одна строка — расходы одного канала за один месяц.
monthly_spend11 = (spend11.groupby(["channel", "month"], as_index=False).agg(total_spend=("amount", "sum")))
# Складываем число регистраций по дням, а не считаем строки таблицы.
monthly_registrations11 = (registrations11.groupby(["channel", "month"], as_index=False).agg(n_clients=("n_clients", "sum")))
print("Расходы по каналам и месяцам:")
display(monthly_spend11.head())
print("Регистрации по каналам и месяцам:")
display(monthly_registrations11.head())

In [ ]:
# Соединяем по двум ключам: канал и месяц.
cac11 = monthly_spend11.merge(monthly_registrations11,on=["channel", "month"],how="left",validate="1:1")
# При отсутствии регистраций или нулевом количестве CAC не определён.
denominator11 = cac11["n_clients"].where(cac11["n_clients"] > 0)
cac11["cac"] = cac11["total_spend"] / denominator11
print("Строк в справочнике CAC:", len(cac11))
print("Строк без рассчитанного CAC:", int(cac11["cac"].isna().sum()))
display(cac11.head(10))

In [ ]:
# Для клиента нужен канал привлечения и месяц его регистрации.
clients11 = clients[["client_id", "acquisition_channel", "registered_at"]].copy()
clients11["month"] = (pd.to_datetime(clients11["registered_at"]).dt.strftime("%Y-%m"))
clients11 = clients11.rename(columns={"acquisition_channel": "channel"})
# Многие клиенты могут иметь одну комбинацию канала и месяца.
mart11 = clients11.merge(cac11[["channel", "month", "cac"]],on=["channel", "month"],how="left",validate="m:1")
print("Клиентов до соединения:", len(clients11))
print("Строк после соединения:", len(mart11))
print("Клиентов с рассчитанным CAC:", int(mart11["cac"].notna().sum()))
print("Клиентов без CAC:", int(mart11["cac"].isna().sum()))
display(mart11.head(10))

In [ ]:
# mean() пропускает отсутствующие значения: нулями их не заменяем.
average_cac11 = mart11["cac"].mean()
missing_cac11 = mart11["cac"].isna().sum()
answer_11 = [round(float(average_cac11), 2),int(missing_cac11)]
print("Средний CAC клиентов сегмента, ₽:", answer_11[0])
print("Клиентов без CAC:", answer_11[1])
answer_11

### Задача 12. Лучшие регионы канала

Где продавать ПРАЙМ дальше? Маркетинг просит три лучших региона по выручке в каждом канале привлечения — и начинает с самого доходного канала.

**Что сделать**

Посчитайте выручку сегмента за окно по каналу и региону владельца; клиенты без канала не участвуют. В каждом канале отберите три региона с наибольшей выручкой. Ответ — три `region_code` самого доходного канала сегмента, от большей выручки к меньшей.

**Формат ответа:** список из трёх `int` — например `[12, 34, 56]`.

**Подсказка:** Слайд 25 Л4: «три лучших вообще» и «три лучших в каждом канале» — разные вещи.

In [ ]:
# Задача 12. Три лучших региона в каждом канале; ответ — для самого доходного канала.
# Ответ — список из трёх int, от большей выручки к меньшей.

from IPython.display import display
# Только успешные платежи приносят выручку.
payments12 = pay[pay["status"] == "success"].copy()
payments12 = payments12.merge(subs[["subscription_id", "owner_client_id"]],on="subscription_id",how="left",
validate="m:1")
print("Успешных платежей:", len(payments12))
display(payments12.head())

In [ ]:
# Каждому платежу присоединяем данные его владельца.
payments12 = payments12.merge(
    clients[["client_id", "acquisition_channel", "region_code"]],
    left_on="owner_client_id",
    right_on="client_id",
    how="left",
    validate="m:1"
)

print("Строк после соединения:", len(payments12))
print(
    "Платежей без канала:",
    int(payments12["acquisition_channel"].isna().sum())
)

# По условию клиенты без канала не участвуют.
payments12 = payments12[
    payments12["acquisition_channel"].notna()
].copy()

print("Платежей с известным каналом:", len(payments12))
display(
    payments12[
        ["owner_client_id", "acquisition_channel", "region_code", "amount"]
    ].head()
)

In [ ]:
# Одна строка — один регион внутри одного канала.
regions12 = (payments12.groupby(["acquisition_channel", "region_code"], as_index=False).agg(revenue=("amount", "sum")))
print("Выручка по каналам и регионам:")
display(regions12.head(10))

In [ ]:
# Внутри канала сортируем по убыванию выручки.
# При равной выручке первым будет меньший код региона.
regions12 = regions12.sort_values(["acquisition_channel", "revenue", "region_code"],ascending=[True, False, True])
top_regions12 = (regions12.groupby("acquisition_channel").head(3))
print("Три лучших региона в каждом канале:")
display(top_regions12)

In [ ]:
# Считаем выручку канала по всем его платежам, а не только по топ-3.
channels12 = (payments12.groupby("acquisition_channel", as_index=False)
    .agg(revenue=("amount", "sum"))
    .sort_values(["revenue", "acquisition_channel"],ascending=[False, True]))
print("Каналы по общей выручке:")
display(channels12)
best_channel12 = channels12.iloc[0]["acquisition_channel"]
print("Самый доходный канал:", best_channel12)

In [ ]:
# Выбираем уже отсортированный топ-3 самого доходного канала.
best_regions12 = top_regions12[top_regions12["acquisition_channel"] == best_channel12]
answer_12 = [int(code)for code in best_regions12["region_code"]]
print("Лучшие регионы выбранного канала:")
display(best_regions12)
print("Ответ:", answer_12)
answer_12

**Вопрос 3.** Куда вы добавили бы рубль на привлечение? Сравните каналы по двум числам: CAC канала (задача 11) и выручка на клиента вашего сегмента из этого канала. Назовите канал и оба числа. Почему выручку вашего сегмента нельзя просто разделить на CAC канала — какую оговорку нужно сделать?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
from IPython.display import display

# Считаем успешную выручку каждого владельца за первое полугодие.
revenue_q3 = (
    pay[pay["status"] == "success"]
    .merge(
        subs[["subscription_id", "owner_client_id"]],
        on="subscription_id",
        validate="m:1"
    )
    .groupby("owner_client_id", as_index=False)
    .agg(paid=("amount", "sum"))
)

# Сохраняем всех клиентов, включая тех, кто не платил.
clients_q3 = mart11[["client_id", "channel", "cac"]].merge(
    revenue_q3,
    left_on="client_id",
    right_on="owner_client_id",
    how="left",
    validate="1:1"
)

# Нет успешных платежей — выручка 0. Неизвестный CAC оставляем пропуском.
clients_q3["paid"] = clients_q3["paid"].fillna(0)

# Для сравнения каналов исключаем клиентов без указанного канала.
known_q3 = clients_q3[clients_q3["channel"].notna()].copy()

comparison_q3 = (
    known_q3.groupby("channel", as_index=False)
    .agg(
        clients_count=("client_id", "size"),
        revenue=("paid", "sum"),
        revenue_per_client=("paid", "mean"),
        clients_with_cac=("cac", "count"),
        average_cac=("cac", "mean")
    )
)

comparison_q3["clients_without_cac"] = (
    comparison_q3["clients_count"] - comparison_q3["clients_with_cac"]
)

# Дополнительно считаем выручку на клиента только среди клиентов с известным CAC.
comparable_q3 = (
    known_q3[known_q3["cac"].notna()]
    .groupby("channel", as_index=False)
    .agg(revenue_per_client_with_cac=("paid", "mean"))
)

comparison_q3 = comparison_q3.merge(
    comparable_q3,
    on="channel",
    how="left",
    validate="1:1"
)

comparison_q3 = comparison_q3.sort_values(
    "revenue_per_client", ascending=False
)

result_q3 = comparison_q3.rename(columns={
    "channel": "Канал",
    "clients_count": "Клиентов сегмента",
    "revenue": "Выручка за полугодие, ₽",
    "revenue_per_client": "Выручка на клиента, ₽",
    "clients_with_cac": "Клиентов с CAC",
    "average_cac": "Средний CAC, ₽",
    "clients_without_cac": "Клиентов без CAC",
    "revenue_per_client_with_cac": "Выручка на клиента с известным CAC, ₽"
})

display(result_q3.round(2))

print(
    "Клиентов без указанного канала:",
    int(clients_q3["channel"].isna().sum())
)

## ✍️ Ответ на вопрос 3
organic - САС 139.34, выручка на клиента 879.08
referral - САС 350.04, выручка на клиента 813.86
crm	- САС 215.51, выручка на клиента 757.16	
partner_telecom	- САС 680.46, выручка на клиента 685.48	
performance	- САС 1504.57, выручка на клиента 567.89
social - САС 794.44	, выручка на клиента 454.72
Рубль я бы добавил в organic из-за низкого САС и Высокой вырчки, но нужно сделать оговорки:
1. САС есть только для 8152	из 24 102 клиентов
2. Выручка расчитана по полугодию, а САС по месяцам, лучше проводить анализ в разрезе одного периода и одной группы клиентов
3. Также не стоит судить по каналу только через призму моего сегмента, при учете других сегментов может получиться другой результат.
4. Повышение вложений в привлечение не всегда дает результат сопоставимый затратам

## Часть 5. Уровень 4 — витрина и протокол сборки

Главная задача недели. Вы собираете **витрину клиента**: одна строка — один клиент сегмента, рядом всё, что про него известно. И доказываете, что двойного счёта нет, **протоколом сборки** — той же формы, что на слайде 15 Л4 и на семинаре С4.

> **Протокол — это 3 балла из 10.** Без него ноль, даже если все числа верные: результат, который нельзя перепроверить, в работе не принимают.

Три последние строки протокола — инварианты витрины:

- **ЗЕРНО:** строк в витрине столько же, сколько уникальных клиентов, и столько, сколько вы объявили до сборки;
- **ДЕНЬГИ:** сумма по витрине сходится с независимым контролем до копейки;
- **ЛЮДИ:** число тех, кто пользуется ПРАЙМ, сходится со счётом, сделанным другим путём.

Ниже готовые функции, эту ячейку просто запустите. `step()` — та же, что на семинаре: одна строка протокола на одно соединение. `check_step()` делает строку протокола для проверки из вашего `src/prime/checks.py`. `print_protocol()` печатает протокол и возвращает ответ задачи 13.

In [ ]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки.

    keys_lost — строки слева, которым не нашлось пары справа.
    Видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost, "verdict": verdict}


def check_step(name: str, frame: pd.DataFrame, bad: pd.DataFrame, expected: int) -> dict:
    """Строка протокола для проверки из checks.py.

    bad — строки, которые нашла проверка; expected — сколько вы ждали, объявив это ДО сборки.
    """
    found = len(bad)
    verdict = f"ok, ожидали {expected}" if found == expected else f"ВНИМАНИЕ: нашлось {found}, ждали {expected}"
    return {"step": name, "rows_before": len(frame), "rows_after": len(frame),
            "validate": "проверка", "keys_lost": found, "verdict": verdict}


def print_protocol(protocol: list[dict], mart: pd.DataFrame, declared: int,
                   money_mart: float, money_control: float,
                   people_mart: int, people_control: int) -> list[int]:
    """Напечатать протокол сборки. Вернуть [строк в витрине, разница ДЕНЬГИ в копейках, ЛЮДИ по витрине]."""
    rows, unique = len(mart), int(mart["client_id"].nunique())
    kopecks_mart, kopecks_control = round(money_mart * 100), round(money_control * 100)
    diff = kopecks_mart - kopecks_control

    print("ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента")
    print()
    print(pd.DataFrame(protocol).to_string(index=False))
    print()
    print(f"ЗЕРНО   строк {rows}, клиентов {unique}, объявляли {declared}"
          f"   -> {'ok' if rows == unique == declared else 'ВНИМАНИЕ'}")
    print(f"ДЕНЬГИ  по витрине {money_mart:.2f} ₽, независимо {money_control:.2f} ₽, разница {diff} коп."
          f"   -> {'ok' if diff == 0 else 'ВНИМАНИЕ'}")
    print(f"ЛЮДИ    по витрине {people_mart}, независимо {people_control}"
          f"   -> {'ok' if people_mart == people_control else 'ВНИМАНИЕ'}")
    print()
    print("HW2_PROTOCOL=" + json.dumps({
        "steps": protocol, "grain": [rows, unique, declared],
        "money_kopecks": [kopecks_mart, kopecks_control], "people": [int(people_mart), int(people_control)],
    }, ensure_ascii=False, default=str))
    return [int(rows), int(diff), int(people_mart)]


print("функции протокола готовы")

### Задача 13. Витрина клиента

Соберите витрину по своему сегменту и докажите протоколом, что двойного счёта нет. Весь раздел 4 семинара С4 — это тот же путь на общем сегменте.

**Что сделать**

1. **До сборки** объявите, сколько строк будет в витрине, и сколько ключей потеряет каждое левое соединение — эти числа пойдут в `expect_lost`.
2. Соберите `mart`, одна строка — один клиент сегмента:
   - `client_id`, `acquisition_channel`, `region_code` — из `clients`;
   - `subscription_id`, `tariff_id` — подписка клиента, пропуск, если её нет;
   - `n_payments`, `paid` — успешные платежи за окно, свёрнутые до клиента; `0`, если их нет;
   - `n_memberships` — в скольких подписках клиент участник; `0`, если ни в одной;
   - `uses_prime` — владелец или участник.
3. Каждое соединение запишите в протокол через `step()`.
4. Примените к витрине функцию из своего `src/prime/checks.py` — например, `find_missing_channel` — и запишите её строкой `check_step()` с ожиданием, объявленным до сборки. Это отдельные 0,5 балла из 10.
5. Посчитайте независимые контроли для ДЕНЬГИ и ЛЮДИ **не по витрине, а другим путём** и вызовите `print_protocol()`.

**Формат ответа:** то, что вернула `print_protocol()`: список из трёх `int`.

**Подсказка:** Если `src/prime/checks.py` у вас нет — создайте его по образцу семинара С3 и закоммитьте в `main`: `from prime.checks import find_missing_channel`. Протокол должен уметь покраснеть: проверьте его, сломав одно соединение, как на семинаре.

In [ ]:
# Задача 13. Витрина клиента и протокол сборки.
# Одна строка — один клиент сегмента. Каждое соединение — через step()
# с ожиданием, объявленным ДО сборки. Проверка из checks.py — через check_step().
# В конце — answer_13 = print_protocol(...).
from prime.checks import find_missing_channel
from IPython.display import display


# 1. ОЖИДАНИЯ И КОНТРОЛЬНЫЕ ЧИСЛА — ДО СБОРКИ

# В выручку включаем только успешные платежи.
success13 = pay[pay["status"] == "success"].copy()

# Множества позволяют считать уникальных клиентов без повторов.
client_ids13 = set(clients["client_id"])
owner_ids13 = set(subs["owner_client_id"])
member_ids13 = set(members["client_id"])

# Находим владельцев подписок, по которым были успешные платежи.
paid_subscription_ids13 = set(success13["subscription_id"])
paying_owner_ids13 = set(
    subs.loc[
        subs["subscription_id"].isin(paid_subscription_ids13),
        "owner_client_id"
    ]
)

# Заранее определяем ожидаемые пропуски после соединений.
declared13 = len(client_ids13)
expected_no_sub13 = len(client_ids13 - owner_ids13)
expected_no_pay13 = len(client_ids13 - paying_owner_ids13)
expected_no_members13 = len(client_ids13 - member_ids13)
expected_missing_channel13 = int(
    clients["acquisition_channel"].isna().sum()
)

# Каждый загруженный платёж должен найти свою подписку.
expected_unmatched_payments13 = 0

# Независимые контроли: считаем по исходным данным, не по витрине.
money_control13 = float(success13["amount"].sum())
people_control13 = len(
    client_ids13 & (owner_ids13 | member_ids13)
)

print("ОЖИДАНИЯ ДО СБОРКИ")
print("Клиентов:", declared13)
print("Без собственной подписки:", expected_no_sub13)
print("Без успешных платежей:", expected_no_pay13)
print("Без участий:", expected_no_members13)
print("Без канала:", expected_missing_channel13)
print("Контроль ДЕНЬГИ, ₽:", round(money_control13, 2))
print("Контроль ЛЮДИ:", people_control13)


# 2. ПОДГОТОВКА ПЛАТЕЖЕЙ И УЧАСТИЙ

# При повторном запуске создаём протокол заново.
protocol = []

# Добавляем владельца: много платежей относятся к одной подписке.
pay_owner13 = success13.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

protocol.append(
    step(
        "платёж -> владелец",
        success13,
        pay_owner13,
        "owner_client_id",
        validate="m:1",
        expect_lost=expected_unmatched_payments13
    )
)

# Сворачиваем платежи до одной строки на клиента.
payments_by_client13 = (
    pay_owner13
    .groupby("owner_client_id", as_index=False)
    .agg(
        n_payments=("payment_id", "size"),
        paid=("amount", "sum")
    )
    .rename(columns={"owner_client_id": "client_id"})
)

# Сворачиваем участия, чтобы они не размножали клиентов при соединении.
memberships13 = (
    members.groupby("client_id", as_index=False)
    .agg(n_memberships=("subscription_id", "size"))
)

print("\nПЛАТЕЖИ ПО КЛИЕНТАМ")
display(payments_by_client13.head())

print("УЧАСТИЯ ПО КЛИЕНТАМ")
display(memberships13.head())


# 3. КЛИЕНТЫ + СОБСТВЕННЫЕ ПОДПИСКИ

# Основа витрины: одна строка — один клиент сегмента.
mart = clients[
    ["client_id", "acquisition_channel", "region_code"]
].copy()

owned13 = subs[
    ["owner_client_id", "subscription_id", "tariff_id"]
].rename(columns={"owner_client_id": "client_id"})

# Сохраняем таблицу до соединения для протокола.
before13 = mart

mart = mart.merge(
    owned13,
    on="client_id",
    how="left",
    validate="1:1"
)

protocol.append(
    step(
        "клиент -> подписка",
        before13,
        mart,
        "subscription_id",
        validate="1:1",
        expect_lost=expected_no_sub13
    )
)

print("\nПОСЛЕ ДОБАВЛЕНИЯ ПОДПИСОК")
print("Строк:", len(mart))
display(mart.head())


# 4. ДОБАВЛЯЕМ ПЛАТЕЖИ

before13 = mart

mart = mart.merge(
    payments_by_client13,
    on="client_id",
    how="left",
    validate="1:1"
)

# Пропуски фиксируем до fillna, пока они ещё видны.
protocol.append(
    step(
        "клиент -> платежи",
        before13,
        mart,
        "n_payments",
        validate="1:1",
        expect_lost=expected_no_pay13
    )
)

# Нет успешных платежей — количество и сумма равны нулю.
mart["n_payments"] = mart["n_payments"].fillna(0).astype("int64")
mart["paid"] = mart["paid"].fillna(0)

print("\nПОСЛЕ ДОБАВЛЕНИЯ ПЛАТЕЖЕЙ")
print("Строк:", len(mart))
display(mart.head())


# 5. ДОБАВЛЯЕМ УЧАСТИЯ

before13 = mart

mart = mart.merge(
    memberships13,
    on="client_id",
    how="left",
    validate="1:1"
)

protocol.append(
    step(
        "клиент -> участия",
        before13,
        mart,
        "n_memberships",
        validate="1:1",
        expect_lost=expected_no_members13
    )
)

# Нет участия — ставим 0.
mart["n_memberships"] = (
    mart["n_memberships"].fillna(0).astype("int64")
)

# Пользователь ПРАЙМ — владелец или участник хотя бы одной подписки.
mart["uses_prime"] = (
    mart["subscription_id"].notna()
    | (mart["n_memberships"] > 0)
)

print("\nГОТОВАЯ ВИТРИНА")
print("Строк:", len(mart))
display(mart.head())


# 6. ПРОВЕРКА ИЗ checks.py

# Проверяем готовую витрину функцией из отдельного файла.
missing_channel13 = find_missing_channel(mart)

protocol.append(
    check_step(
        "проверка: нет канала",
        mart,
        missing_channel13,
        expected=expected_missing_channel13
    )
)


# 7. ИТОГОВЫЙ ПРОТОКОЛ: ЗЕРНО, ДЕНЬГИ, ЛЮДИ

# Сравниваем витрину с контролями, рассчитанными до её сборки.
answer_13 = print_protocol(
    protocol=protocol,
    mart=mart,
    declared=declared13,
    money_mart=float(mart["paid"].sum()),
    money_control=money_control13,
    people_mart=int(mart["uses_prime"].sum()),
    people_control=people_control13
)

answer_13

**Вопрос 4.** Где ваш протокол заранее ждал потерь ключей и почему это не ошибка — назовите числа. Что стало бы со строками ЗЕРНО и ЛЮДИ, если бы участия не свернули до клиента? Посчитайте, а не угадывайте.

In [ ]:
# Числа для ответа на вопрос 4 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
# 1. Ожидаемые пропуски из протокола правильной сборки.
print("ОЖИДАЕМЫЕ ПРОПУСКИ")
print("Клиентов без собственной подписки:", expected_no_sub13)
print("Клиентов без успешных платежей:", expected_no_pay13)
print("Клиентов без участий:", expected_no_members13)
print("Клиентов без канала:", expected_missing_channel13)

# 2. Берём копию витрины до добавления сведений об участиях.
before_q4 = mart.drop(columns=["n_memberships", "uses_prime"]).copy()

# Оставляем каждое участие отдельной строкой — намеренно без groupby.
raw_members_q4 = members[["client_id", "subscription_id"]].rename(
    columns={"subscription_id": "membership_subscription_id"}
)

# Правильная проверка 1:1 должна заметить повторяющихся клиентов справа.
print("\nПРОВЕРКА КРАТНОСТИ")
try:
    before_q4.merge(
        raw_members_q4,
        on="client_id",
        how="left",
        validate="1:1"
    )
    print("Проверка 1:1 прошла: повторяющихся клиентов в участиях нет.")
except pd.errors.MergeError:
    print("Проверка 1:1 обнаружила несколько участий одного клиента.")

# 3. Для эксперимента разрешаем несколько строк справа.
# Это намеренно неправильная сборка витрины «одна строка — один клиент».
broken_q4 = before_q4.merge(
    raw_members_q4,
    on="client_id",
    how="left",
    validate="1:m"
)

broken_q4["uses_prime"] = (
    broken_q4["subscription_id"].notna()
    | broken_q4["membership_subscription_id"].notna()
)

# 4. Сравниваем число строк и подсчёт пользователей.
rows_q4 = len(broken_q4)
unique_q4 = int(broken_q4["client_id"].nunique())
people_q4 = int(broken_q4["uses_prime"].sum())

print("\nЗЕРНО БЕЗ СВОРАЧИВАНИЯ УЧАСТИЙ")
print("Ожидали строк:", declared13)
print("Получили строк:", rows_q4)
print("Уникальных клиентов:", unique_q4)
print("Лишних строк:", rows_q4 - declared13)
print(
    "Результат:",
    "ok" if rows_q4 == unique_q4 == declared13 else "ВНИМАНИЕ"
)

print("\nЛЮДИ БЕЗ СВОРАЧИВАНИЯ УЧАСТИЙ")
print("Независимый контроль:", people_control13)
print("Сумма признака uses_prime:", people_q4)
print("Завышение:", people_q4 - people_control13)
print(
    "Результат:",
    "ok" if people_q4 == people_control13 else "ВНИМАНИЕ"
)

# Уникальные люди сохраняются, но некоторые повторяются в строках.
unique_users_q4 = broken_q4.loc[
    broken_q4["uses_prime"], "client_id"
].nunique()

print("Уникальных пользователей при отдельном подсчёте:", int(unique_users_q4))

#### ✍️ Ответ на вопрос 4

keys_lost выдал числа, которые показывают кому не нашлось соответствующей записи справа при объединении слева:
30 558 - без собственной подписки
57 373 - без успешных платежей
33 135 - без участий
Если бы участия не свернули до клиента, то рассчитанное количество пользователей завысилось бы на 44 610 клиентов, увеличилось число строк с 101 036 до 145 646, зерно нарушилось.

## Часть 6. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 14)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» после каждого уровня.")
print("\n--- служебная строка, не удаляйте ---")
print("HW2_ANSWERS=" + json.dumps(
    {"client_filter": CLIENT_FILTER, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 7. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–6 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: в нём две части, каждая до 1 балла сверху. Они засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется у преподавателя целиком — в том числе на контрольном сегменте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Загруженных таблиц для исследования хватает: `pay`, `subs` и `tariffs`. Считайте от них, а не от чисел, вписанных руками, — на контрольном сегменте код должен отработать без правок.

### Исследование А. Что подорожание дало выручке

С 1 марта 2026 года «Базовый» подорожал с 199 до 249 ₽, «Расширенный» — с 399 до 449 ₽. Финансы сравнили выручку сегмента за март–июнь с январём–февралём и записали весь рост в заслугу подорожания. Руководитель просит проверить, сколько рублей в месяц на самом деле дала цена.

**Что сделать**

1. По месяцам окна посчитайте выручку и число успешных платежей — по каждому тарифу.
2. Разделите изменение выручки после 1 марта на две части. **Цена** — разница между тем, сколько стоили платежи марта–июня, и тем, сколько те же платежи стоили бы по старой цене. **Объём** — всё остальное.
3. Учтите две вещи, которые меняют число платежей в месяц без всякого подорожания. Подписка списывается раз в 30 дней, а месяцы разной длины. И в данных апреля есть дыра — вы видели её в вопросе 2.
4. Ответьте: сколько рублей в месяц приносит подорожание вашему сегменту — и насколько ошибается наивное сравнение «март–июнь против января–февраля».

In [ ]:
# Исследование А: ваш код. Всё — от загруженных таблиц.
from IPython.display import display

# 1. ГОТОВИМ УСПЕШНЫЕ ПЛАТЕЖИ ЗА ОКНО ИССЛЕДОВАНИЯ

start_a = pd.Timestamp("2026-01-01", tz="UTC")
change_a = pd.Timestamp("2026-03-01", tz="UTC")
end_a = pd.Timestamp("2026-07-01", tz="UTC")

# Все попытки нужны для проверки полноты дат.
all_pay_a = pay[
    (pay["paid_at"] >= start_a)
    & (pay["paid_at"] < end_a)
].copy()

all_pay_a["day"] = all_pay_a["paid_at"].dt.normalize()
all_pay_a["month"] = all_pay_a["paid_at"].dt.strftime("%Y-%m")

# Выручку считаем только по успешным платежам.
payments_a = all_pay_a[
    all_pay_a["status"] == "success"
].merge(
    subs[["subscription_id", "tariff_id"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)


# 2. ПОЛУЧАЕМ СТАРЫЕ ЦЕНЫ ИЗ СПРАВОЧНИКА

ref_a = tariffs.copy()

for col in ("valid_from", "valid_to"):
    ref_a[col] = pd.to_datetime(ref_a[col], utc=True)

# Берём цены, действовавшие на начало января.
# Ошибочная строка 429 сюда не попадёт: она начинается в феврале.
old_prices_a = ref_a.loc[
    (ref_a["valid_from"] <= start_a)
    & (start_a < ref_a["valid_to"] + pd.Timedelta(days=1)),
    ["tariff_id", "tariff_name", "monthly_fee"]
].rename(columns={"monthly_fee": "old_price"})

print("ЦЕНЫ ДО ПОДОРОЖАНИЯ")
display(old_prices_a)

payments_a = payments_a.merge(
    old_prices_a,
    on="tariff_id",
    how="left",
    validate="m:1"
)

# У каждого платежа должна быть найдена старая цена.
assert payments_a["old_price"].notna().all(), "Не найдена старая цена"

# Стоимость тех же платежей при сохранении старых цен.
payments_a["revenue_old_price"] = payments_a["old_price"]


# 3. ПРОВЕРЯЕМ ПОЛНОТУ ДАТ

calendar_a = pd.date_range(
    start=start_a,
    end=end_a - pd.Timedelta(days=1),
    freq="D"
)

missing_days_a = calendar_a.difference(
    pd.DatetimeIndex(all_pay_a["day"].unique())
)

# Число дат с любыми попытками списания, а не только успешными.
observed_days_a = (
    all_pay_a.groupby("month")["day"]
    .nunique()
)

month_starts_a = pd.date_range(
    start=start_a,
    end=end_a,
    freq="MS",
    inclusive="left"
)

coverage_a = pd.DataFrame({
    "month": month_starts_a.strftime("%Y-%m"),
    "calendar_days": month_starts_a.days_in_month
})

coverage_a["observed_days"] = (
    coverage_a["month"].map(observed_days_a).fillna(0).astype(int)
)

print("\nПОЛНОТА ДАННЫХ")
display(coverage_a)
print(
    "Отсутствующие даты:",
    missing_days_a.strftime("%Y-%m-%d").tolist()
)


# 4. ВЫРУЧКА И ЧИСЛО ПЛАТЕЖЕЙ ПО МЕСЯЦАМ И ТАРИФАМ

monthly_a = (
    payments_a.groupby(["month", "tariff_id"])
    .agg(
        n_payments=("payment_id", "size"),
        revenue=("amount", "sum"),
        revenue_old_price=("revenue_old_price", "sum")
    )
)

# Добавляем все комбинации месяцев и тарифов, включая нулевые платежи.
grid_a = pd.MultiIndex.from_product(
    [coverage_a["month"], old_prices_a["tariff_id"]],
    names=["month", "tariff_id"]
)

monthly_a = monthly_a.reindex(grid_a, fill_value=0).reset_index()

monthly_a = monthly_a.merge(
    old_prices_a[["tariff_id", "tariff_name"]],
    on="tariff_id",
    how="left",
    validate="m:1"
).merge(
    coverage_a,
    on="month",
    how="left",
    validate="m:1"
)

# Не делим на ноль, если за целый месяц нет записей.
days_a = monthly_a["observed_days"].where(
    monthly_a["observed_days"] > 0
)

monthly_a["payments_per_30_days"] = (
    monthly_a["n_payments"] / days_a * 30
)
monthly_a["revenue_per_30_days"] = (
    monthly_a["revenue"] / days_a * 30
)

print("\nПО МЕСЯЦАМ И ТАРИФАМ")
display(monthly_a[[
    "month", "tariff_name", "n_payments", "revenue",
    "calendar_days", "observed_days",
    "payments_per_30_days", "revenue_per_30_days"
]].round(2))


# 5. РАЗЛОЖЕНИЕ ИЗМЕНЕНИЯ: ЦЕНА И ОБЪЁМ

before_a = monthly_a[monthly_a["month"] < "2026-03"]
after_a = monthly_a[monthly_a["month"] >= "2026-03"]

factor_rows_a = []

for tariff_id_a in old_prices_a["tariff_id"]:
    before_t_a = before_a[before_a["tariff_id"] == tariff_id_a]
    after_t_a = after_a[after_a["tariff_id"] == tariff_id_a]

    tariff_name_a = old_prices_a.loc[
        old_prices_a["tariff_id"] == tariff_id_a, "tariff_name"
    ].iloc[0]

    # Наивное сравнение средних календарных месяцев.
    naive_before_a = before_t_a["revenue"].sum() / len(before_t_a)
    naive_after_a = after_t_a["revenue"].sum() / len(after_t_a)
    naive_growth_a = naive_after_a - naive_before_a

    # Для сопоставимого периода используем суммарные наблюдаемые дни.
    before_days_a = int(before_t_a["observed_days"].sum())
    after_days_a = int(after_t_a["observed_days"].sum())

    if before_days_a == 0 or after_days_a == 0:
        raise ValueError("Недостаточно дат для сравнения периодов")

    before_30_a = before_t_a["revenue"].sum() / before_days_a * 30
    after_30_a = after_t_a["revenue"].sum() / after_days_a * 30

    # Те же платежи после подорожания, но по старым ценам.
    after_old_30_a = (
        after_t_a["revenue_old_price"].sum() / after_days_a * 30
    )

    price_effect_a = after_30_a - after_old_30_a
    volume_effect_a = after_old_30_a - before_30_a
    total_change_a = after_30_a - before_30_a

    factor_rows_a.append({
        "Тариф": tariff_name_a,
        "До, ₽ / 30 дней": float(before_30_a),
        "После, ₽ / 30 дней": float(after_30_a),
        "Изменение, ₽ / 30 дней": float(total_change_a),
        "Цена, ₽ / 30 дней": float(price_effect_a),
        "Объём, ₽ / 30 дней": float(volume_effect_a),
        "Наивный рост, ₽ / календарный месяц": float(naive_growth_a)
    })

factors_a = pd.DataFrame(factor_rows_a)

print("\nВЛИЯНИЕ ЦЕНЫ И ОБЪЁМА")
display(factors_a.round(2))


# 6. ОБЩИЙ РЕЗУЛЬТАТ И ПРОВЕРКА РАЗЛОЖЕНИЯ

price_total_a = factors_a["Цена, ₽ / 30 дней"].sum()
volume_total_a = factors_a["Объём, ₽ / 30 дней"].sum()
change_total_a = factors_a["Изменение, ₽ / 30 дней"].sum()
naive_total_a = factors_a["Наивный рост, ₽ / календарный месяц"].sum()

# Факторы должны в сумме дать изменение выручки.
balance_a = round(
    float(change_total_a - price_total_a - volume_total_a), 2
)

print("\nИТОГ")
print("Влияние цены, ₽ на 30 дней:", round(float(price_total_a), 2))
print("Влияние объёма, ₽ на 30 дней:", round(float(volume_total_a), 2))
print("Общее изменение, ₽ на 30 дней:", round(float(change_total_a), 2))
print("Проверка: изменение − цена − объём =", balance_a)

# Наивный вывод финансов: весь рост средней месячной выручки — заслуга цены.
print(
    "\nНаивно приписали цене, ₽ в среднем за календарный месяц:",
    round(float(naive_total_a), 2)
)
print(
    "Наивная оценка минус эффект цены на 30 дней, ₽:",
    round(float(naive_total_a - price_total_a), 2)
)

# Отдельно показываем эффект цены без пересчёта на отсутствующие дни.
observed_price_effect_a = float(
    after_a["revenue"].sum() - after_a["revenue_old_price"].sum()
)

print(
    "\nЭффект цены по имеющимся платежам марта–июня, всего ₽:",
    round(observed_price_effect_a, 2)
)
print(
    "В среднем на календарный месяц марта–июня, без восстановления пропусков, ₽:",
    round(
        observed_price_effect_a / after_a["month"].nunique(),
        2
    )
)

#### ✍️ Исследование А

1. Необходимо выяснить вклад изменения стоимости подписки на рост выручки.
2. а) рассчитал количество и выручку по успешным платежам за месяц по тарифам
   б) проверил все ли данные есть по датам
   в) привел показатели к 30 дням, чтобы сравнивать периоды одной длительности. (за 30 дней,которые в выгрузке есть записи о попытках списания)
   г) для марта-июня расчитал выручку по старым ценам подписки
   д) рассчитал разницу между фактической выручкой и с старой ценой (эффект цены) и эффект из-за изменения объема (эффект объема)
3. Что получилось: Эффект цены на 30 дней - +1 824 925 руб., Эффект объема на 30 дней - −1 210 991,21 руб., Общее изменение на 30 дней - +613 933,79 руб., Наивное изменение средней календарной месячной выручки - +801 699,50 руб.
4. Результат - повышение цены увеличило выручку, однако в рассматриваемом разрезе времени платежей стало меньше, что частично перекрыло рост. Поэтому изменение общей выручки нельзя целиком преписывать цене. Наивное сравнение занижает вклад повышения цены на 1 023 225,50 руб (1 824 925 − 801 699,50). Снижение количества платежей частично скрывает положительный эффект цены.
5. Ограничения: Пропущенные дни апреля (14 и 15 число), пересчет на опирается на среднее по наблюдаемым дням.

### Исследование Б. Стоило ли подорожание подписчиков

Маркетинг опасается, что подорожание вызвало волну отмен, и предлагает вернуть старые цены. Проверьте на своём сегменте, есть ли для этого основания.

**Что сделать**

1. По месяцам с января по июнь посчитайте отток: сколько подписок закончилось в месяце из тех, что были активны на его начало. Данные — `subs`, колонки `started_at` и `ended_at`.
2. Сравните отток до и после 1 марта — отдельно для «Базового» (+25 %) и «Расширенного» (+12,5 %). Если бы цена отпугивала, какой тариф должен был отреагировать сильнее?
3. Посмотрите на причины отмен: как менялась доля `too_expensive` среди отмен по месяцам.
4. Вывод для маркетинга: есть ли в данных след подорожания в оттоке — и чего ваш расчёт не доказывает. Сколько у вас отмен в месяц и хватит ли этого, чтобы заметить эффект?

In [ ]:
# Исследование Б: ваш код. Всё — от загруженных таблиц.
from IPython.display import display

# 1. ПОДГОТОВКА ДАННЫХ

subs_b = subs.copy()

for col in ("started_at", "ended_at"):
    subs_b[col] = pd.to_datetime(subs_b[col], utc=True)

# Для названий тарифов история цен не нужна.
names_b = tariffs[["tariff_id", "tariff_name"]].drop_duplicates()

subs_b = subs_b.merge(
    names_b,
    on="tariff_id",
    how="left",
    validate="m:1"
)

# Границы шести месяцев: январь — июнь.
boundaries_b = pd.date_range(
    "2026-01-01", "2026-07-01", freq="MS", tz="UTC"
)

rows_b = []


# 2. ОТТОК И ПРИЧИНЫ ОТМЕН ПО МЕСЯЦАМ И ТАРИФАМ

for start_b, end_b in zip(boundaries_b[:-1], boundaries_b[1:]):
    for tariff_id_b in names_b["tariff_id"]:
        tariff_subs_b = subs_b[
            subs_b["tariff_id"] == tariff_id_b
        ]

        tariff_name_b = names_b.loc[
            names_b["tariff_id"] == tariff_id_b, "tariff_name"
        ].iloc[0]

        # Как на лекции: подписка началась до начала месяца
        # и ещё не закончилась к его началу.
        active_b = tariff_subs_b[
            (tariff_subs_b["started_at"] < start_b)
            & (
                tariff_subs_b["ended_at"].isna()
                | (tariff_subs_b["ended_at"] >= start_b)
            )
        ]

        # Отток: закончившиеся за месяц ИЗ активных на его начало.
        churned_b = active_b[
            active_b["ended_at"] < end_b
        ]

        # Для анализа причин берём ВСЕ отмены за месяц.
        # Здесь могут быть и подписки, начавшиеся в этом же месяце.
        cancelled_b = tariff_subs_b[
            (tariff_subs_b["ended_at"] >= start_b)
            & (tariff_subs_b["ended_at"] < end_b)
        ]

        expensive_b = int(
            (cancelled_b["cancel_reason"] == "too_expensive").sum()
        )
        missing_reason_b = int(
            cancelled_b["cancel_reason"].isna().sum()
        )

        rows_b.append({
            "Месяц": start_b.strftime("%Y-%m"),
            "Период": "До" if start_b.month < 3 else "После",
            "Тариф": tariff_name_b,
            "Активны на начало": len(active_b),
            "Ушли из активных": len(churned_b),
            "Всего отмен": len(cancelled_b),
            "Отмен too_expensive": expensive_b,
            "Отмен без причины": missing_reason_b
        })

monthly_b = pd.DataFrame(rows_b)

# При нулевом знаменателе показатель остаётся неопределённым.
monthly_b["Отток, %"] = (
    monthly_b["Ушли из активных"]
    / monthly_b["Активны на начало"].where(
        monthly_b["Активны на начало"] > 0
    )
    * 100
)

monthly_b["Доля too_expensive, %"] = (
    monthly_b["Отмен too_expensive"]
    / monthly_b["Всего отмен"].where(monthly_b["Всего отмен"] > 0)
    * 100
)

print("ОТТОК И ПРИЧИНЫ ОТМЕН ПО МЕСЯЦАМ")
display(monthly_b[[
    "Месяц", "Тариф",
    "Активны на начало", "Ушли из активных", "Отток, %",
    "Всего отмен", "Отмен too_expensive",
    "Доля too_expensive, %", "Отмен без причины"
]].round(2))


# 3. СРАВНЕНИЕ ДО И ПОСЛЕ ПОДОРОЖАНИЯ

summary_b = (
    monthly_b.groupby(["Тариф", "Период"], as_index=False)
    .agg(
        months=("Месяц", "size"),
        active_sum=("Активны на начало", "sum"),
        churned_sum=("Ушли из активных", "sum"),
        cancellations=("Всего отмен", "sum"),
        expensive=("Отмен too_expensive", "sum"),
        missing_reason=("Отмен без причины", "sum")
    )
)

# Средний месячный отток, взвешенный по числу активных подписок.
# Это НЕ доля ушедших за весь двух- или четырёхмесячный период.
summary_b["Отток, %"] = (
    summary_b["churned_sum"]
    / summary_b["active_sum"].where(summary_b["active_sum"] > 0)
    * 100
)

# Доля причины среди всех отмен соответствующего периода.
summary_b["Доля too_expensive, %"] = (
    summary_b["expensive"]
    / summary_b["cancellations"].where(summary_b["cancellations"] > 0)
    * 100
)

summary_b["Отмен в среднем за месяц"] = (
    summary_b["cancellations"] / summary_b["months"]
)

print("\nСРАВНЕНИЕ ЯНВАРЯ–ФЕВРАЛЯ И МАРТА–ИЮНЯ")

display(
    summary_b.rename(columns={
        "months": "Месяцев",
        "active_sum": "Сумма активных на начала месяцев",
        "churned_sum": "Ушли из активных",
        "cancellations": "Всего отмен",
        "expensive": "Отмен too_expensive",
        "missing_reason": "Отмен без причины"
    }).round(2)
)


# 4. ИЗМЕНЕНИЯ В ПРОЦЕНТНЫХ ПУНКТАХ

print("\nИЗМЕНЕНИЯ ПОСЛЕ 1 МАРТА")

for tariff_name_b in summary_b["Тариф"].unique():
    tariff_summary_b = (
        summary_b[summary_b["Тариф"] == tariff_name_b]
        .set_index("Период")
    )

    churn_before_b = tariff_summary_b.loc["До", "Отток, %"]
    churn_after_b = tariff_summary_b.loc["После", "Отток, %"]

    expensive_before_b = tariff_summary_b.loc[
        "До", "Доля too_expensive, %"
    ]
    expensive_after_b = tariff_summary_b.loc[
        "После", "Доля too_expensive, %"
    ]

    print("\nТариф:", tariff_name_b)
    print("Отток до, %:", round(float(churn_before_b), 2))
    print("Отток после, %:", round(float(churn_after_b), 2))
    print(
        "Изменение оттока, п.п.:",
        round(float(churn_after_b - churn_before_b), 2)
    )
    print(
        "Изменение доли too_expensive, п.п.:",
        round(float(expensive_after_b - expensive_before_b), 2)
    )


# 5. ЧИСЛО ОТМЕН: МАСШТАБ ДАННЫХ ДЛЯ ВЫВОДА

cancel_counts_b = monthly_b.pivot_table(
    index="Месяц",
    columns="Тариф",
    values="Всего отмен",
    aggfunc="sum",
    fill_value=0
)

print("\nСКОЛЬКО ОТМЕН В КАЖДОМ МЕСЯЦЕ")
display(cancel_counts_b)

print("\nСКОЛЬКО ОТМЕН С ПРИЧИНОЙ too_expensive")
display(
    monthly_b.pivot_table(
        index="Месяц",
        columns="Тариф",
        values="Отмен too_expensive",
        aggfunc="sum",
        fill_value=0
    )
)

#### ✍️ Исследование Б

1. Проверил вырос ли отток после повышения цен на 1 марта. Какой тариф сильнее отрегировал? Есть ли смысл возвращать старые цены?
2. а) Для каждого месяца января-июня по каждому тарифу определил подписки активные на начало месяца
   б) Посчитал сколько закончилось в этом месяце, разделил на число активных и получил месячный отток
   в) Сравнил янвварь-февраль с мартом-июнем, использовал средний месячный отток, взвешенный по числу активных подписок
   г) Отдельно среди отмененных подписок посчитал долю причины (too_expensive) и вывел количество отмен по месяцам.
3. Что получилось: Базовая подписка - отток уменьшился с 3,86% до 3,79%, доля too_expensive выросла с 23,32% до 24,29%, отмен месяц 810 – 1 057. Расширенная подписка - отток с 3,90% до 3,69%, доля too_expensive выросла с 23,46% до 24,65%, отмен месяц 463 – 701.
4. Общий отток после марта не вырос. Доля отмены из-за дороговизны немного увеличилась, но это само по себе не доказывает волну отмены из-за цены. Расчет не дает достаточных оснований возвращать старые цены именно из-за предпологаемого оттока. Сравнение не доказывает ни причинного влияния цены, ни отсутствия такого влияния